In [1]:
import geopandas as gpd
import networkx as nx
import numpy as np
import pickle
from shapely.geometry import LineString
from geopy.distance import geodesic
from scipy.spatial import KDTree

In [2]:
FILE_PATH = "../data/raw/OGIM_v1.1.gpkg"
LAYER = "Oil_Natural_Gas_Pipelines"


In [3]:
MINX, MAXX = 68, 97
MINY, MAXY = 8, 37

In [5]:
print("Loading dataset...")
pipelines = gpd.read_file(FILE_PATH, layer=LAYER, ignore_geometry=True)

print("Filtering region...")
pipelines = gpd.read_file(
    FILE_PATH,
    layer=LAYER,
    bbox=(68, 8, 97, 37)   # (minx, miny, maxx, maxy)
)

print(f"Remaining pipelines: {len(pipelines)}")


Loading dataset...
Filtering region...
Remaining pipelines: 123


In [ ]:
print("Simplifying geometries...")
pipelines["geometry"] = pipelines.geometry.simplify(tolerance=0.01)


In [6]:
pipelines.to_file("india_pipelines.geojson", driver="GeoJSON")

In [7]:
gpd.read_file("india_pipelines.geojson")

,OGIM_ID,CATEGORY,REGION,COUNTRY,STATE_PROV,SRC_REF_ID,SRC_DATE,ON_OFFSHORE,FAC_NAME,FAC_ID,...,OGIM_STATUS,OPERATOR,INSTALL_DATE,COMMODITY,GAS_CAPACITY_MMCFD,GAS_THROUGHPUT_MMCFD,PIPE_DIAMETER_MM,PIPE_LENGTH_KM,PIPE_MATERIAL,geometry
0,OGIM_5915761,OIL AND NATURAL GAS PIPELINES,EURASIA,INDIA,N/A,22,2013-01-01,"ONSHORE, OFFSHORE",N/A,N/A,...,N/A,N/A,1900-01-01,N/A,-999.0,-999,-999.0,1380.0,N/A,"LINESTRING (72.6356 21.11285, 73.18204 22.3049..."
1,OGIM_5915890,OIL AND NATURAL GAS PIPELINES,EURASIA,INDIA,N/A,22,2014-01-01,"ONSHORE, OFFSHORE",N/A,N/A,...,N/A,N/A,1900-01-01,N/A,-999.0,-999,-999.0,208.0,N/A,"LINESTRING (76.22448 9.98452, 78.08202 10.95653)"
2,OGIM_5915892,OIL AND NATURAL GAS PIPELINES,EURASIA,INDIA,N/A,22,2014-01-01,"ONSHORE, OFFSHORE",N/A,N/A,...,N/A,N/A,1900-01-01,N/A,-999.0,-999,-999.0,259.0,N/A,"LINESTRING (74.81279 12.91726, 76.09181 13.007..."
3,OGIM_5915893,OIL AND NATURAL GAS PIPELINES,EURASIA,INDIA,N/A,22,2014-01-01,ONSHORE,N/A,N/A,...,N/A,N/A,1900-01-01,N/A,-999.0,-999,-999.0,122.0,N/A,"LINESTRING (79.08894 11.40669, 78.95003 11.533..."
4,OGIM_5915757,OIL AND NATURAL GAS PIPELINES,EURASIA,INDIA,N/A,22,2013-01-01,"ONSHORE, OFFSHORE",N/A,N/A,...,N/A,N/A,1900-01-01,N/A,-999.0,-999,-999.0,259.0,N/A,"LINESTRING (74.81279 12.91726, 76.09181 13.007..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
118,OGIM_5915903,OIL AND NATURAL GAS PIPELINES,EURASIA,INDIA,N/A,22,2014-01-01,ONSHORE,N/A,N/A,...,N/A,N/A,1900-01-01,N/A,-999.0,-999,-999.0,368.0,N/A,"LINESTRING (75.57896 31.32102, 76.16991 31.194..."
119,OGIM_5916045,OIL AND NATURAL GAS PIPELINES,ASIA PACIFIC,PAKISTAN,N/A,164,2014-04-01,ONSHORE,SNGPL SYSTEM,N/A,...,N/A,SUI NORTHERN GAS PIPELINES LIMITED,1900-01-01,N/A,-999.0,-999,-999.0,103.0,N/A,"LINESTRING (74.28477 31.88268, 74.13486 31.861..."
120,OGIM_5916046,OIL AND NATURAL GAS PIPELINES,ASIA PACIFIC,PAKISTAN,N/A,164,2014-04-01,ONSHORE,SNGPL SYSTEM,N/A,...,N/A,SUI NORTHERN GAS PIPELINES LIMITED,1900-01-01,N/A,-999.0,-999,-999.0,520.0,N/A,"LINESTRING (73.70596 33.27466, 73.63458 33.167..."
121,OGIM_5916047,OIL AND NATURAL GAS PIPELINES,ASIA PACIFIC,PAKISTAN,N/A,164,2014-04-01,ONSHORE,SNGPL SYSTEM,N/A,...,N/A,SUI NORTHERN GAS PIPELINES LIMITED,1900-01-01,N/A,-999.0,-999,-999.0,980.0,N/A,"LINESTRING (71.49306 34.08368, 71.60014 33.976..."


In [11]:
import geopandas as gpd

FILE_PATH = "../data/raw/OGIM_v1.1.gpkg"

print("Loading wells safely...")

wells = gpd.read_file(
    FILE_PATH,
    layer="Oil_and_Natural_Gas_Wells",
    bbox=(68, 8, 97, 37),
    engine="fiona"
)

# Fix duplicate geometry issue
wells = wells.loc[:, ~wells.columns.duplicated()]

# Clean
wells = wells.dropna(subset=["geometry"])
wells = wells[wells.is_valid]

# Keep only geometry
wells = wells[["geometry"]]

# Save
wells.to_file("india_wells.geojson", driver="GeoJSON")

print("✅ Wells extracted successfully!")

Loading wells safely...
✅ Wells extracted successfully!


In [13]:
import geopandas as gpd

FILE_PATH = "../data/raw/OGIM_v1.1.gpkg"

wells = gpd.read_file(
    FILE_PATH,
    layer="Oil_and_Natural_Gas_Wells",
    bbox=(68, 8, 97, 37),
    engine="fiona"
)

# Remove duplicate columns issue
wells = wells.loc[:, ~wells.columns.duplicated()]

# Clean
wells = wells.dropna(subset=["geometry"])
wells = wells[wells.is_valid]

# ✅ KEEP IMPORTANT COLUMNS
keep_cols = ["geometry", "COUNTRY", "STATE_PROV", "NAME", "OGIM_ID"]

# Only keep columns that exist
keep_cols = [col for col in keep_cols if col in wells.columns]

wells = wells[keep_cols]

# Save
wells.to_file("india_wells1.geojson", driver="GeoJSON")

print("✅ Wells with attributes saved")

✅ Wells with attributes saved


In [18]:
import geopandas as gpd

wells = gpd.read_file("../notebooks/india_wells1.geojson")
print(wells["COUNTRY"].unique())

['AFGHANISTAN']


In [ ]:
import geopandas as gpd

FILE_PATH = "../data/raw/OGIM_v1.1.gpkg"

print("Loading wells from OGIM...")

wells = gpd.read_file(
    FILE_PATH,
    layer="Oil_and_Natural_Gas_Wells",
    engine="fiona"
)

# -------------------------------
# CLEAN DATA
# -------------------------------
wells = wells.loc[:, ~wells.columns.duplicated()]
wells = wells.dropna(subset=["geometry"])
wells = wells[wells.is_valid]

print("Countries available:")
print(wells["COUNTRY"].unique())

# -------------------------------
# FILTER INDIA (CORRECT WAY)
# -------------------------------
wells_india = wells[
    wells["COUNTRY"].astype(str).str.contains("IND", case=False, na=False)
]

print("India wells count:", len(wells_india))

# -------------------------------
# SAVE CORRECT FILE
# -------------------------------
wells_india.to_file("data/india_wells.geojson", driver="GeoJSON")

print("✅ Correct India wells saved")

Loading wells from OGIM...
